# Week 2 — Day 2: Baseline evaluation (BLEU / ROUGE)

Tasks:
- Set up evaluation metrics for captioning: BLEU, ROUGE
- Run zero-shot BLIP over a larger validation subset (200-500 images) and compute metric scores

Deliverable: baseline evaluation report (metric scores table) for zero-shot BLIP on the Flickr8k
validation subset.

### 1. Load BLIP and pick a 300-image validation subset

In [1]:
import os
import time
import pandas as pd
import torch
import evaluate
from PIL import Image
from tqdm import tqdm
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name).to(device)
model.eval()

captions_path = '../data/captions.txt'
img_dir = '../data/Images'

df = pd.read_csv(captions_path)
df.columns = ['image', 'caption']

VAL_SIZE = 300
val_images = df['image'].drop_duplicates().sample(VAL_SIZE, random_state=123).tolist()
# all 5 ground-truth captions per image, for multi-reference scoring
references = [df[df['image'] == img]['caption'].tolist() for img in val_images]
print(f"Validation subset: {len(val_images)} images, {sum(len(r) for r in references)} reference captions")

Using device: cpu


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Validation subset: 300 images, 1500 reference captions


### 2. Generate zero-shot captions for the whole subset

In [2]:
@torch.no_grad()
def generate_caption(img_path):
    image = Image.open(img_path).convert('RGB')
    inputs = processor(images=image, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=30)
    return processor.decode(out[0], skip_special_tokens=True)

start = time.time()
predictions = []
for img_name in tqdm(val_images):
    pred = generate_caption(os.path.join(img_dir, img_name))
    predictions.append(pred)
elapsed = time.time() - start
print(f"Generated {len(predictions)} captions in {elapsed:.1f}s ({elapsed/len(predictions):.2f}s/image)")

  0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 1/300 [00:03<17:47,  3.57s/it]

  1%|          | 2/300 [00:07<18:07,  3.65s/it]

  1%|          | 3/300 [00:10<17:26,  3.53s/it]

  1%|▏         | 4/300 [00:13<16:07,  3.27s/it]

...

 99%|█████████▉| 298/300 [16:00<00:05,  2.86s/it]

100%|█████████▉| 299/300 [16:03<00:02,  2.81s/it]

100%|██████████| 300/300 [16:05<00:00,  2.79s/it]

100%|██████████| 300/300 [16:05<00:00,  3.22s/it]

Generated 300 captions in 965.8s (3.22s/image)


### 3. Compute BLEU and ROUGE against the 5 human references per image

In [3]:
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")

bleu_result = bleu_metric.compute(predictions=predictions, references=references)
rouge_result = rouge_metric.compute(predictions=predictions, references=references)

scores_table = pd.DataFrame([
    {"metric": "BLEU (sacrebleu)", "score": bleu_result["score"]},
    {"metric": "ROUGE-1", "score": rouge_result["rouge1"] * 100},
    {"metric": "ROUGE-2", "score": rouge_result["rouge2"] * 100},
    {"metric": "ROUGE-L", "score": rouge_result["rougeL"] * 100},
])
scores_table

,metric,score
0,BLEU (sacrebleu),16.656270
1,ROUGE-1,50.194672
2,ROUGE-2,26.643420
3,ROUGE-L,48.080152


### 4. Save the report

In [4]:
os.makedirs('../data/processed', exist_ok=True)
scores_table.to_csv('../data/processed/week2_baseline_eval_scores.csv', index=False)

report_df = pd.DataFrame({
    "image": val_images,
    "predicted_caption": predictions,
    "reference_captions": ["; ".join(r) for r in references],
})
report_df.to_csv('../data/processed/week2_baseline_eval_predictions.csv', index=False)
print("Saved scores and per-image predictions to data/processed/")
report_df.head(10)

Saved scores and per-image predictions to data/processed/


,image,predicted_caption,reference_captions
0,2304374703_555195d8d5.jpg,a dog playing with a ball in the snow,A brown dog plays with a bright orange ball in...
1,3497238310_2abde3965d.jpg,a man and a boy playing basketball in a gym,A group of men stand in a gymnasium with a bas...
2,3434526008_02359881a0.jpg,a skateboarder is doing a trick on a ramp,A man in a sleeveless shirt is performing an a...
3,3349258288_5300c40430.jpg,a man on a skateboard,A guy gettin some air on a skateboard; A man r...
4,369360998_ba56fb436f.jpg,a woman kneeling in the snow,A blonde woman with a red backpack in the snow...
5,3501206996_477be0f318.jpg,a young boy playing soccer,A child in a brown coat is kicking a soccer ba...
6,3677329561_fa3e1fdcf9.jpg,a dog with a frumbent,A dog is being washed by two little girls .; t...
7,1917265421_aeccf1ca38.jpg,a group of people,Children participate in a sport involving swin...
8,2819254573_9ecb5f4d5e.jpg,a boy in a chair,A child is yelling whilst on a roller coaster;...
9,3327563443_870a33f748.jpg,a woman wearing a hat,A woman in a white baseball hat reveals her ta...


### Notes

- Scores are computed zero-shot (no fine-tuning yet) — this is the **baseline** the Week 2 Day 3-4
  fine-tuning experiments need to beat.
- BLEU/ROUGE against 5 free-form human references per image are inherently noisy (valid captions can
  describe an image very differently and still score low n-gram overlap), so treat these as a
  reference point rather than an absolute quality measure — the qualitative review from Day 1 matters
  just as much.
- This same generation + scoring loop is reused in Day 4 to compare fine-tuning configurations.